# MKDT sur CIFAR-10


## Montage Drive (persistance)

In [ ]:
import torch, subprocess
print("GPU dispo:", torch.cuda.is_available())
print(subprocess.getoutput("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader"))

GPU dispo: True
NVIDIA A100-SXM4-40GB, 40960 MiB


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
PERSIST = "/content/drive/MyDrive/mkdt_project"
os.makedirs(PERSIST, exist_ok=True)
print("Persistance:", PERSIST)

Mounted at /content/drive
Persistance: /content/drive/MyDrive/mkdt_project


## Clone du dépôt officiel et les dépendances

In [ ]:
%cd /content
![ -d MKDT ] || git clone https://github.com/BigML-CS-UCLA/MKDT.git
%cd /content/MKDT
!pip install -q wandb
import os
os.environ["WANDB_MODE"] = "offline"
print("OK")

/content
Cloning into 'MKDT'...
remote: Enumerating objects: 237, done.
remote: Counting objects: 100% (18/18), done.
remote: Compressing objects: 100% (8/8), done.
remote: Total 237 (delta 13), reused 10 (delta 10), pack-reused 219 (from 1)
Receiving objects: 100% (237/237), 13.35 MiB | 30.51 MiB/s, done.
Resolving deltas: 100% (77/77), done.
/content/MKDT
OK


## Écriture des fichiers patchés



In [ ]:
%%writefile train_teacher_barlow_twins.py

import os
import argparse
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms
from torchvision.models import resnet18


def build_backbone():
    m = resnet18()
    m.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=2, bias=False)
    m.maxpool = nn.Identity()
    m.fc = nn.Identity()
    return m


class BarlowTwins(nn.Module):
    def __init__(self, proj_dim=1024):
        super().__init__()
        self.backbone = build_backbone()
        self.projector = nn.Sequential(
            nn.Linear(512, proj_dim, bias=False), nn.BatchNorm1d(proj_dim), nn.ReLU(inplace=True),
            nn.Linear(proj_dim, proj_dim, bias=False), nn.BatchNorm1d(proj_dim), nn.ReLU(inplace=True),
            nn.Linear(proj_dim, proj_dim, bias=False),
        )
        self.bn = nn.BatchNorm1d(proj_dim, affine=False)

    def forward(self, y1, y2):
        z1 = self.projector(self.backbone(y1))
        z2 = self.projector(self.backbone(y2))
        c = self.bn(z1).T @ self.bn(z2)
        c.div_(y1.size(0))
        return c


def barlow_twins_loss(c, lambd=0.0051):
    on_diag = torch.diagonal(c).add_(-1).pow_(2).sum()
    off_diag = off_diagonal(c).pow_(2).sum()
    return on_diag + lambd * off_diag


def off_diagonal(x):
    n, _ = x.shape
    return x.flatten()[:-1].view(n - 1, n + 1)[:, 1:].flatten()


class TwoViews:
    def __init__(self, size, mean, std):
        self.t = transforms.Compose([
            transforms.RandomResizedCrop(size, scale=(0.2, 1.0)),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomApply([transforms.ColorJitter(0.4, 0.4, 0.2, 0.1)], p=0.8),
            transforms.RandomGrayscale(p=0.2),
            transforms.GaussianBlur(kernel_size=3),
            transforms.ToTensor(),
            transforms.Normalize(mean=mean, std=std),
        ])

    def __call__(self, x):
        return self.t(x), self.t(x)


def get_train_dataset(name, data_path):
    if name == "CIFAR10":
        mean, std, size = [0.4914, 0.4822, 0.4465], [0.2023, 0.1994, 0.2010], 32
        ds = datasets.CIFAR10(data_path, train=True, download=True,
                              transform=TwoViews(size, mean, std))
    elif name == "CIFAR100":
        mean, std, size = [0.5071, 0.4865, 0.4409], [0.2009, 0.1984, 0.2023], 32
        ds = datasets.CIFAR100(data_path, train=True, download=True,
                               transform=TwoViews(size, mean, std))
    else:
        raise ValueError(f"Dataset {name} non géré par ce script (CIFAR10 / CIFAR100).")
    return ds


def main(args):
    device = f"cuda:{args.device}" if torch.cuda.is_available() else "cpu"
    ds = get_train_dataset(args.dataset, args.data_path)
    dl = torch.utils.data.DataLoader(ds, batch_size=args.batch_size, shuffle=True,
                                     num_workers=args.workers, pin_memory=True, drop_last=True)

    model = BarlowTwins(proj_dim=args.proj_dim).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=args.lr, weight_decay=1e-6)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=args.epochs)

    model.train()
    for epoch in range(1, args.epochs + 1):
        total = 0.0
        for (y1, y2), _ in dl:
            y1, y2 = y1.to(device, non_blocking=True), y2.to(device, non_blocking=True)
            c = model(y1, y2)
            loss = barlow_twins_loss(c, lambd=args.lambd)
            opt.zero_grad()
            loss.backward()
            opt.step()
            total += loss.item()
        sched.step()
        print(f"Epoch {epoch:4d}/{args.epochs} | loss {total / len(dl):.4f} | lr {sched.get_last_lr()[0]:.5f}")

    os.makedirs(args.out, exist_ok=True)
    ckpt_name = f"barlow_twins_resnet18_{args.dataset.lower()}.pt"
    out_path = os.path.join(args.out, ckpt_name)
    torch.save(model.backbone.state_dict(), out_path)
    print(f"\nCheckpoint teacher sauvegardé : {out_path}")
    print("Il est directement chargeable par get_target_rep.py.")


if __name__ == "__main__":
    p = argparse.ArgumentParser()
    p.add_argument("--dataset", type=str, default="CIFAR10", choices=["CIFAR10", "CIFAR100"])
    p.add_argument("--data_path", type=str, default="./data")
    p.add_argument("--out", type=str, default="./krrst_teacher_ckpt")
    p.add_argument("--epochs", type=int, default=800)
    p.add_argument("--batch_size", type=int, default=256)
    p.add_argument("--lr", type=float, default=0.01)
    p.add_argument("--lambd", type=float, default=0.0051)
    p.add_argument("--proj_dim", type=int, default=1024)
    p.add_argument("--workers", type=int, default=8)
    p.add_argument("--device", type=int, default=0)
    main(p.parse_args())


Writing train_teacher_barlow_twins.py


In [ ]:
%%writefile get_target_rep.py
import os
import argparse
import torch
import torch.nn as nn
from tqdm import tqdm
from utils import get_dataset
from torchvision.models import resnet18

def main(args):
    args.device = f'cuda:{args.device}' if torch.cuda.is_available() else 'cpu'
    _, _, _, dst_train, _ = get_dataset(dataset=args.dataset, data_path=args.data_path)

    images_all = []

    print("BUILDING TRAINSET")
    for i in tqdm(range(len(dst_train))):
        sample = dst_train[i]
        images_all.append(torch.unsqueeze(sample[0], dim=0))

    images_all = torch.cat(images_all, dim=0).to("cpu")

    output_dir = os.path.join(args.result_dir, args.ssl_algorithm)
    os.makedirs(output_dir, exist_ok=True)

    labels_all = []
    print(images_all.shape)
    if args.ssl_algorithm == "barlow_twins":
        target_model = resnet18()
        target_model.fc = nn.Identity()
        target_model.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=2, bias=False)
        target_model.maxpool = nn.Identity()
        target_model = target_model.to(args.device)
        ds_tag = args.dataset.lower() if args.dataset != 'Tiny' else 'tinyimagenet'
        checkpoint = torch.load(f"{args.teacher_ckpt_dir}/barlow_twins_resnet18_{ds_tag}.pt", map_location="cpu")
        keys_to_remove = ["fc.weight", "fc.bias"]
        for key in keys_to_remove:
            checkpoint.pop(key, None)
    else:
        from resnet import ResNet18, StemCIFAR
        target_model = ResNet18(stem=StemCIFAR)
        loaded_model = torch.load(f"/home/sjoshi/sas-data-efficient-contrastive-learning/{args.data_name}-resnet18-net.pt", map_location="cpu")
        checkpoint = loaded_model.state_dict()

    target_model.load_state_dict(checkpoint)
    target_model.eval()

    with torch.no_grad():
        for image in images_all:
            image = image.to(args.device).unsqueeze(0)
            image_repr = target_model(image)
            labels_all.append(image_repr)

    labels_all = torch.cat(labels_all, dim=0)

    torch.save(labels_all.detach().cpu(), f"{args.result_dir}/{args.ssl_algorithm}/{args.dataset}_target_rep_train.pt")

    print("train label shape", labels_all.shape)

if __name__ == '__main__':
    parser = argparse.ArgumentParser(description='Get Target Representation Parameter Processing')
    parser.add_argument('--dataset', type=str, default='CIFAR100', help='dataset')
    parser.add_argument('--data_path', type=str, default='/home/data', help='dataset path')
    parser.add_argument('--result_dir', type=str, default='target_rep', help='dataset path')
    parser.add_argument('--device', type=int, default=0, help='gpu number')
    parser.add_argument('--ssl_algorithm', type=str, default='barlow_twins', choices=['barlow_twins', 'simclr'], help='SSL Algorithm used to get the target representation.')
    parser.add_argument('--teacher_ckpt_dir', type=str, default='./krrst_teacher_ckpt', help='dossier contenant le checkpoint teacher')
    parser.add_argument('--model', type=str, default='ConvNet', help='(non utilisé ici, gardé pour compat CLI)')

    args = parser.parse_args()
    main(args)





Overwriting get_target_rep.py


In [ ]:
%%writefile sort_dset.py
import argparse
import torch
import os
from tqdm import tqdm
from utils import get_dataset, get_network, build_trainset
from tqdm import tqdm
import pickle
from reparam_module import ReparamModule
import numpy as np


def main(args):
    channel, im_size, _, dst_train, _ = get_dataset(dataset=args.dataset, data_path=args.data_path)
    train_labels_path = f"./target_rep/{args.ssl_algo}/{args.dataset}_target_rep_train.pt"

    trainloader, labels_all = build_trainset(dataset=args.dataset,
                                 dst_train=dst_train,
                                 train_labels_path=train_labels_path,
                                 channel=channel,
                                 batch_train=args.batch_train,
                                 shuffle=False
                                )
    epoch_losses = {}

    num_examples = len(trainloader.dataset)
    print(num_examples)
    epoch_losses = np.zeros(num_examples)

    for num_buffer in range(1, args.num_buffers + 1):
        checkpoint_path = f'./buffers_{args.ssl_algo}/{args.dataset}/{args.model}/replay_buffer_{num_buffer-1}.pt'
        param_all = torch.load(checkpoint_path)
        first_epoch_checkpoint = param_all[0][1]
        first_epoch_checkpoint = torch.cat([p.data.to(args.device).reshape(-1) for p in first_epoch_checkpoint], 0)

        teacher_net = get_network(args.model, channel, labels_all.shape[1], im_size).to(args.device)
        teacher_net = ReparamModule(teacher_net)

        for i, (inputs, targets) in enumerate(tqdm(trainloader)):
            inputs, targets = inputs.to(args.device), targets.to(args.device)
            with torch.no_grad():
                outputs = teacher_net(inputs, flat_param=first_epoch_checkpoint)
                mse_losses = torch.mean((outputs - targets) ** 2, dim=1)
                individual_losses = mse_losses.detach().cpu().numpy()
                epoch_losses[i * args.batch_train:(i + 1) * args.batch_train] += individual_losses

    epoch_losses /= args.num_buffers

    sorted_indices = np.argsort(epoch_losses)[::-1]

    print("Sorted indices of high loss subset:", sorted_indices)

    dset_name = (args.dataset).lower() if args.dataset != "Tiny" else "tiny_imagenet"

    if args.dataset != "ImageNet":
        top_2_percent_idx = sorted_indices[:int(0.02 * num_examples)]
        top_5_percent_idx = sorted_indices[:int(0.05 * num_examples)]

        output_dir = f'./init/{dset_name}'
        os.makedirs(output_dir, exist_ok=True)

        with open(f'{output_dir}/{args.dataset}_{args.ssl_algo}_2_high_loss_indices.pkl', 'wb') as f:
            pickle.dump(top_2_percent_idx, f)
        with open(f'{output_dir}/{args.dataset}_{args.ssl_algo}_5_high_loss_indices.pkl', 'wb') as f:
            pickle.dump(top_5_percent_idx, f)

    else:
        top_1000 = sorted_indices[:1000]

        output_dir = f'./init/{dset_name}'
        os.makedirs(output_dir, exist_ok=True)

        with open(f'{output_dir}/{args.dataset}_{args.ssl_algo}_1000_high_loss_indices.pkl', 'wb') as f:
            pickle.dump(top_1000, f)

if __name__ == '__main__':
    parser = argparse.ArgumentParser(description='Sort Dataset Parameter Processing')
    parser.add_argument('--dataset', type=str, default='CIFAR100', help='dataset')
    parser.add_argument('--model', type=str, default='ConvNet', help='model')
    parser.add_argument('--num_buffers', type=int, default=100, help='number of buffers')
    parser.add_argument('--ssl_algo', type=str, default="barlow_twins", choices=['barlow_twins', 'simclr'], help='Algorithm to train the SSL')
    parser.add_argument('--batch_train', type=int, default=256, help='batch size for training networks')
    parser.add_argument('--data_path', type=str, default='/home/data', help='dataset path')
    parser.add_argument('--device', type=int, default=0, help='gpu number')
    args = parser.parse_args()
    main(args)

Overwriting sort_dset.py


In [ ]:
%%writefile eval.py
import os
import random
import argparse
import pickle
import wandb
import numpy as np
import torch
import pandas as pd
from torch.utils.data import TensorDataset, DataLoader
from torchvision.utils import make_grid
from utils import get_dataset, get_network, SoftLabelDataset
from pretrain_methods import pretrain_mse
from evaluation.linear_evaluation import le_run

def aggregate_results(results):
    aggregated_results = {}
    for dataset in results[0].keys():
        dataset_results = np.array([result[dataset] for result in results])
        mean = np.mean(dataset_results, axis=0)
        std = np.std(dataset_results, axis=0)
        aggregated_results[dataset] = {'mean': mean.tolist(), 'std': std.tolist()}

    return aggregated_results

def main(args):
    target_datasets = [d.strip() for d in args.test_datasets.split(",")]

    wandb.init(
        project="mkdt_data_distillation_evaluation",
        config=args
    )

    final_res = {}
    for td in target_datasets:
        args.test_dataset = td
        res_list = []
        #device
        device = torch.device(f"cuda:{args.device}")
        torch.cuda.set_device(device)

        # seed
        random.seed(args.seed)
        np.random.seed(args.seed)
        torch.manual_seed(args.seed)

        # data
        train_img_channel, train_img_size, num_pretrain_classes, dst_train_pretrain, _ = get_dataset(dataset=args.train_dataset, data_path=args.data_path)
        eval_img_channel, eval_img_size, eval_num_classes, eval_dst_train, eval_dst_test = get_dataset(args.test_dataset, args.data_path, subset_size=args.subset_frac, epc=args.epc)

        # Used for test
        dl_tr = torch.utils.data.DataLoader(eval_dst_train, batch_size=args.test_batch_size, shuffle=False, num_workers=4, pin_memory=True)
        dl_te = torch.utils.data.DataLoader(eval_dst_test, batch_size=args.test_batch_size, shuffle=False, num_workers=4, pin_memory=True)

        print("Test training data: ", len(eval_dst_train))

        labels_all = torch.load(args.label_path, map_location="cpu")

        if args.result_dir is None:
            if args.subset_path is not None:
                with open(args.subset_path, "rb") as f:
                    subset_idx = pickle.load(f)

            else:
                subset_idx = None

            dst_train_pretrain = SoftLabelDataset(dst_train_pretrain, labels_all, subset_idx)
            print(f"Number of images for pretrain: {len(dst_train_pretrain)}")
            dl_syn = torch.utils.data.DataLoader(dst_train_pretrain, batch_size=args.pre_batch_size, shuffle=True, num_workers=0, pin_memory=True)

            # visualize
            all_images = torch.stack([img for img, _ in dst_train_pretrain], dim=0)
            grid = make_grid(all_images.clone(), nrow=100)
            wandb.log({"undistilled images": wandb.Image(grid.detach().cpu())})
            del all_images


        else:
            try:
                if args.use_krrst:
                    x_syn = torch.load(f"{args.result_dir}/x_syn.pt")
                    x_syn = x_syn.detach().cpu()
                else:
                    x_syn = torch.load(f"{args.result_dir}/images_{args.distilled_steps}.pt")
                    x_syn = x_syn.detach().cpu()
            except:
                raise ValueError("No synthetic images found. ")
            try:
                syn_lr = torch.load(f"{args.result_dir}/synlr_{args.distilled_steps}.pt")
                args.pre_lr = syn_lr.detach().cpu()
            except:
                print("No syn lr found, fall back to original pre_lr")

            try:
                if args.use_krrst:
                    y_syn = torch.load(f"{args.result_dir}/y_syn.pt")
                    y_syn = y_syn.detach().cpu()
                else:
                    y_syn = torch.load(f"{args.result_dir}/labels_{args.distilled_steps}.pt")
                    y_syn = y_syn.detach().cpu()
            except:
                raise ValueError("No synthetic representations found. ")

            assert x_syn.requires_grad == False and y_syn.requires_grad == False
            assert x_syn.grad is None and y_syn.grad is None
            assert y_syn.shape[-1] == labels_all.shape[1]

            x_syn, y_syn = x_syn.detach(), y_syn.detach()
            x_syn, y_syn = x_syn.to(device), y_syn.to(device)

            dst_train_pretrain = TensorDataset(x_syn, y_syn)
            print(f"Number of images for pretrain: {len(dst_train_pretrain)}")
            dl_syn = DataLoader(dst_train_pretrain, batch_size=args.pre_batch_size, shuffle=True, num_workers=0)

            grid = make_grid(x_syn.clone(), nrow=100)
            wandb.log({"distilled images": wandb.Image(grid.detach().cpu())})

        num_target_features = labels_all.shape[1]
        print("Pretraining LR: ", args.pre_lr)

        if args.no_pretrain:
            random_model = get_network(args.train_model, eval_img_channel, num_target_features, train_img_size, fix_net=True).to(device)
            rd_acc = le_run(
                        train_model=args.train_model,
                        channel=eval_img_channel,
                        num_classes=eval_num_classes,
                        img_size=eval_img_size,
                        device=device,
                        init_model=random_model,
                        dl_tr=dl_tr,
                        dl_te=dl_te,
                        le_iters=args.le_iters,
                        seed=args.seed
                    )
            wandb.log({f"{td}_random_accuracy": rd_acc})
            del random_model
            res_list.append(rd_acc)

        else:
            final_model = pretrain_mse(
                                    train_model=args.train_model,
                                    channel=train_img_channel,
                                    num_classes=num_target_features,
                                    img_size=train_img_size,
                                    device=device,
                                    dl_syn=dl_syn,
                                    pre_opt=args.pre_opt,
                                    pre_lr=args.pre_lr,
                                    pre_wd=args.pre_wd,
                                    pre_epoch=args.pre_epoch,
                                )
            final_acc = le_run(
                                train_model=args.train_model,
                                channel=eval_img_channel,
                                num_classes=eval_num_classes,
                                img_size=eval_img_size,
                                device=device,
                                init_model=final_model,
                                dl_tr=dl_tr,
                                dl_te=dl_te,
                                le_iters=args.le_iters,
                                seed=args.seed
                            )

            del final_model
            wandb.log({f"{td}_subset_final_accuracy": final_acc})
            res_list.append(final_acc)

        final_res[td] = res_list

    print(final_res)
    wandb.log(final_res)
    wandb.finish(quiet=True)

    return final_res

if __name__ == '__main__':
    parser = argparse.ArgumentParser(description='Eval Parameter Processing')

    parser.add_argument('--seed', type=int, default=0)
    parser.add_argument('--device', type=int, default=0)

    parser.add_argument('--data_path', type=str, default="/data")
    parser.add_argument('--train_dataset', type=str, default="CIFAR100")
    parser.add_argument('--test_dataset', type=str, default="CIFAR100")

    parser.add_argument('--label_path', type=str, required=True)

    parser.add_argument('--subset_path', type=str, default=None)
    parser.add_argument('--result_dir', type=str, default=None)

    parser.add_argument('--train_model', type=str, default="ConvNet")

    parser.add_argument("--results_csv", default="mkdt_results.csv")

    parser.add_argument('--pre_opt', type=str, default="sgd")
    parser.add_argument('--pre_epoch', type=int, default=20)
    parser.add_argument('--pre_batch_size', type=int, default=256)
    parser.add_argument('--pre_lr', type=float, default=0.1)
    parser.add_argument('--pre_wd', type=float, default=1e-4)
    parser.add_argument('--distilled_steps', type=int, default=1000)
    parser.add_argument('--use_krrst', action="store_true")
    parser.add_argument('--no_pretrain', action="store_true")

    parser.add_argument('--le_iters', type=int, default=20)
    parser.add_argument('--test_batch_size', type=int, default=256)
    parser.add_argument('--subset_frac', type=float, default=None) # fraction of the subset
    parser.add_argument('--epc', type=int, default=None) # examples per class
    parser.add_argument('--test_datasets', type=str, default="CIFAR10", help='tâches downstream, séparées par des virgules')
    parser.add_argument('--seeds', type=str, default="0,1,2", help='seeds séparés par des virgules')

    base_args = parser.parse_args()
    seeds = [int(s) for s in str(base_args.seeds).split(",")]
    results = []

    for seed in seeds:
        args = parser.parse_args()
        args.seed = seed
        results.append(main(args))

    final_aggregated_results = aggregate_results(results)
    print(final_aggregated_results)

    curr_results_df = pd.DataFrame(index=[0])
    curr_results_df["timestamp"] = pd.Timestamp.now()
    args_dict = vars(args)
    for key in args_dict:
        try:
            curr_results_df[key] = args_dict[key]
        except:
            print(f"Couldn't save {key}")

    for dataset, stats in final_aggregated_results.items():
        mean = stats['mean'][0]
        std = stats['std'][0]
        print(f"{dataset}: {mean:.2f} ± {std:.2f}")
        curr_results_df[dataset] = "$" + f"{mean:.2f}" + "\scriptstyle{ \pm " + f"{std:.2f}" + "}" + "$"

    print(curr_results_df)
    if os.path.exists(args.results_csv):
        results_df = pd.read_csv(args.results_csv)
    else:
        results_df = pd.DataFrame()

    results_df = pd.concat([results_df, curr_results_df], ignore_index=True)
    results_df.to_csv(args.results_csv, index=False)

    wandb.finish(quiet=True)

Overwriting eval.py


## Configuration

In [ ]:
DATASET="CIFAR10"; MODEL="ConvNet"; DATA="/content/data"; DEVICE=0; SSL="barlow_twins"
NUM_EXPERTS=30
SEEDS="0"

import os
TEACHER_DIR = f"{PERSIST}/krrst_teacher_ckpt"
REP_DIR     = f"{PERSIST}/target_rep"
BUF_DIR     = f"{PERSIST}/buffers_{SSL}"
LOG_DIR     = f"{PERSIST}/logged_files"
for d in [TEACHER_DIR, REP_DIR, BUF_DIR, LOG_DIR, DATA]: os.makedirs(d, exist_ok=True)

LABELS=f"{REP_DIR}/{SSL}/{DATASET}_target_rep_train.pt"
EXPERT_DIR=f"{BUF_DIR}/{DATASET}/{MODEL}"
INIT_IDX=f"/content/MKDT/init/cifar10/{DATASET}_{SSL}_2_high_loss_indices.pkl"
TEACHER_CKPT=f"{TEACHER_DIR}/barlow_twins_resnet18_{DATASET.lower()}.pt"
print("config OK")

config OK


## Entrainement du Teacher



In [ ]:
if os.path.exists(TEACHER_CKPT):
    print("Teacher présent sur le Drive")
else:
    EPOCHS = 100
    !python train_teacher_barlow_twins.py --dataset {DATASET} --data_path {DATA} \
        --out {TEACHER_DIR} --epochs {EPOCHS} --device {DEVICE}

100% 170M/170M [00:02<00:00, 64.6MB/s]
Epoch    1/100 | loss 587.3304 | lr 0.01000
Epoch    2/100 | loss 470.8889 | lr 0.00999
Epoch    3/100 | loss 387.8575 | lr 0.00998
Epoch    4/100 | loss 338.0637 | lr 0.00996
Epoch    5/100 | loss 295.0317 | lr 0.00994
Epoch    6/100 | loss 266.5707 | lr 0.00991
Epoch    7/100 | loss 244.2239 | lr 0.00988
Epoch    8/100 | loss 227.7331 | lr 0.00984
Epoch    9/100 | loss 211.1634 | lr 0.00980
Epoch   10/100 | loss 197.5289 | lr 0.00976
Epoch   11/100 | loss 186.1251 | lr 0.00970
Epoch   12/100 | loss 175.4442 | lr 0.00965
Epoch   13/100 | loss 167.0203 | lr 0.00959
Epoch   14/100 | loss 159.3368 | lr 0.00952
Epoch   15/100 | loss 152.7563 | lr 0.00946
Epoch   16/100 | loss 145.2509 | lr 0.00938
Epoch   17/100 | loss 139.8005 | lr 0.00930
Epoch   18/100 | loss 135.2678 | lr 0.00922
Epoch   19/100 | loss 130.3670 | lr 0.00914
Epoch   20/100 | loss 126.0078 | lr 0.00905
Epoch   21/100 | loss 122.4132 | lr 0.00895
Epoch   22/100 | loss 119.0184 | lr 0

In [ ]:
!python get_target_rep.py --dataset {DATASET} --model {MODEL} --ssl_algorithm {SSL} \
    --data_path {DATA} --result_dir {REP_DIR} --teacher_ckpt_dir {TEACHER_DIR} --device {DEVICE}
import os; print("labels:", os.path.exists(LABELS))

BUILDING TRAINSET
100% 50000/50000 [00:12<00:00, 3925.65it/s]
torch.Size([50000, 3, 32, 32])
train label shape torch.Size([50000, 512])
labels: True


## Éval CONDITION 1 — Full Data


In [ ]:
!python eval.py --train_dataset {DATASET} --label_path {LABELS} \
    --test_datasets CIFAR10 --subset_frac 0.01 --pre_epoch 20 --seeds {SEEDS} \
    --data_path {DATA} --device {DEVICE} --results_csv {PERSIST}/results_fulldata.csv

/content/MKDT/eval.py:249: SyntaxWarning: invalid escape sequence '\s'
  curr_results_df[dataset] = "$" + f"{mean:.2f}" + "\scriptstyle{ \pm " + f"{std:.2f}" + "}" + "$"
wandb: Tracking run with wandb version 0.27.2
wandb: W&B syncing is set to `offline` in this directory. Run `wandb online` or set WANDB_MODE=online to enable cloud syncing.
wandb: Run data is saved locally in /content/MKDT/wandb/offline-run-20260615_132253-4dttjiou
Test training data:  500
Number of images for pretrain: 50000
wandb: WARNING Data passed to `wandb.Image` should consist of values in the range [0, 255], image data will be normalized to this range, but behavior will be removed in a future version of wandb.
Pretraining LR:  0.1
100% 20/20 [04:07<00:00, 12.39s/it]
encoding: 100% 2/2 [00:00<00:00,  8.84it/s]
torch.Size([500, 2048])
torch.Size([500])

L2 Regularization weight: 0.001
Loss: **** | Train Acc: ****% :   0% 0/20 [00:00<?, ?it/s]/content/MKDT/evaluation/linear_evaluation.py:33: UserWarning: Convertin

## Trajectoires expertes


In [ ]:
import shutil, glob, os
if os.path.exists(EXPERT_DIR): shutil.rmtree(EXPERT_DIR); print("buffers nettoyés")
!python buffer.py --dataset {DATASET} --model {MODEL} --num_experts {NUM_EXPERTS} \
    --buffer_path {BUF_DIR} --train_labels_path {LABELS} \
    --train_epochs 20 --lr_teacher 0.1 --mom 0.9 --l2 1e-4 --criterion mse --device {DEVICE}
print("buffers:", len(glob.glob(f"{EXPERT_DIR}/replay_buffer_*.pt")))

Hyper-parameters: 
 {'dataset': 'CIFAR10', 'model': 'ConvNet', 'num_experts': 30, 'lr_teacher': 0.1, 'batch_train': 256, 'data_path': '/home/data', 'buffer_path': '/content/drive/MyDrive/mkdt_project/buffers_barlow_twins', 'train_epochs': 20, 'mom': 0.9, 'l2': 0.0001, 'train_labels_path': '/content/drive/MyDrive/mkdt_project/target_rep/barlow_twins/CIFAR10_target_rep_train.pt', 'criterion': 'mse', 'device': 'cuda:0'}
100% 170M/170M [00:02<00:00, 67.1MB/s]
BUILDING TRAINSET
100% 50000/50000 [00:12<00:00, 3908.69it/s]
train label shape torch.Size([50000, 512])
real images channel 0, mean = -0.0000, std = 1.2211
real images channel 1, mean = -0.0002, std = 1.2211
real images channel 2, mean = 0.0002, std = 1.3014
Dataset creation complete
ConvNet(
  (features): Sequential(
    (0): Conv2d(3, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): GroupNorm(128, 128, eps=1e-05, affine=True)
    (2): ReLU(inplace=True)
    (3): AvgPool2d(kernel_size=2, stride=2, padding=0)
    (4):

## Distillation MKDT


In [ ]:
!python distill.py --dataset {DATASET} --model {MODEL} --iters 5000 --eval_it 1000 \
    --train_labels_path {LABELS} --expert_dir {EXPERT_DIR} --image_init_idx_path {INIT_IDX} \
    --expert_epochs 2 --syn_steps 40 --max_start_epoch 2 \
    --lr_img 1000 --lr_lr 1e-4 --lr_teacher 0.1 --data_path {DATA}

import glob, os, shutil
runs = sorted(glob.glob(f"/content/MKDT/logged_files/{DATASET}/*"), key=os.path.getmtime)
assert runs, "aucun run trouvé"
RUN_LOCAL = runs[-1]
RUN_DIR = os.path.join(LOG_DIR, os.path.basename(RUN_LOCAL))
shutil.copytree(RUN_LOCAL, RUN_DIR, dirs_exist_ok=True)
print("RUN_DIR (Drive):", RUN_DIR)

Image size: (32, 32)
wandb: Tracking run with wandb version 0.27.2
wandb: W&B syncing is set to `offline` in this directory. Run `wandb online` or set WANDB_MODE=online to enable cloud syncing.
wandb: Run data is saved locally in /content/MKDT/wandb/offline-run-20260615_134509-xzruf3d5
Hyper-parameters: 
 {'_wandb': {'m': [], 'cli_version': '0.27.2', 'python_version': '3.12.13', 't': {'1': [1, 41], '3': [4, 16], '4': '3.12.13', '5': '0.27.2', '12': '0.27.2', '13': 'linux-x86_64'}}, 'dataset': 'CIFAR10', 'model': 'ConvNet', 'eval_it': 1000, 'iters': 5000, 'lr_img': 1000, 'lr_lr': 0.0001, 'lr_teacher': 0.1, 'batch_syn': 256, 'data_path': '/content/data', 'expert_epochs': 2, 'syn_steps': 40, 'max_start_epoch': 2, 'lr_labels': 1, 'image_init_idx_path': '/content/MKDT/init/cifar10/CIFAR10_barlow_twins_2_high_loss_indices.pkl', 'train_labels_path': '/content/drive/MyDrive/mkdt_project/target_rep/barlow_twins/CIFAR10_target_rep_train.pt', 'run_id': None, 'expert_dir': '/content/drive/MyDrive/

## Éval CONDITION 2 — MKDT


In [ ]:
!python eval.py --train_dataset {DATASET} --label_path {LABELS} \
    --result_dir {RUN_DIR} --distilled_steps 5000 \
    --test_datasets CIFAR10 --subset_frac 0.01 --pre_epoch 20 --seeds {SEEDS} \
    --data_path {DATA} --device {DEVICE} --results_csv {PERSIST}/results_mkdt.csv

/content/MKDT/eval.py:249: SyntaxWarning: invalid escape sequence '\s'
  curr_results_df[dataset] = "$" + f"{mean:.2f}" + "\scriptstyle{ \pm " + f"{std:.2f}" + "}" + "$"
wandb: Tracking run with wandb version 0.27.2
wandb: W&B syncing is set to `offline` in this directory. Run `wandb online` or set WANDB_MODE=online to enable cloud syncing.
wandb: Run data is saved locally in /content/MKDT/wandb/offline-run-20260615_151124-4yw5n7is
Test training data:  500
Number of images for pretrain: 1000
wandb: WARNING Data passed to `wandb.Image` should consist of values in the range [0, 255], image data will be normalized to this range, but behavior will be removed in a future version of wandb.
Pretraining LR:  tensor(0.4718)
100% 20/20 [00:00<00:00, 24.78it/s]
encoding: 100% 2/2 [00:00<00:00, 10.23it/s]
torch.Size([500, 2048])
torch.Size([500])

L2 Regularization weight: 0.001
Loss: **** | Train Acc: ****% :   0% 0/20 [00:00<?, ?it/s]/content/MKDT/evaluation/linear_evaluation.py:33: UserWarning:

## Baselines random & no-pretrain

In [ ]:
import numpy as np, pickle, os
os.makedirs("/content/MKDT/init/cifar10", exist_ok=True)
pickle.dump(np.random.choice(50000,1000,replace=False),
            open("/content/MKDT/init/cifar10/cifar10_random.pkl","wb"))
!python eval.py --no_pretrain --train_dataset {DATASET} --label_path {LABELS} \
    --test_datasets CIFAR10 --subset_frac 0.01 --pre_epoch 20 --seeds {SEEDS} \
    --data_path {DATA} --device {DEVICE} --results_csv {PERSIST}/results_nopretrain.csv
!python eval.py --subset_path /content/MKDT/init/cifar10/cifar10_random.pkl \
    --train_dataset {DATASET} --label_path {LABELS} \
    --test_datasets CIFAR10 --subset_frac 0.01 --pre_epoch 20 --seeds {SEEDS} \
    --data_path {DATA} --device {DEVICE} --results_csv {PERSIST}/results_random.csv

/content/MKDT/eval.py:249: SyntaxWarning: invalid escape sequence '\s'
  curr_results_df[dataset] = "$" + f"{mean:.2f}" + "\scriptstyle{ \pm " + f"{std:.2f}" + "}" + "$"
wandb: Tracking run with wandb version 0.27.2
wandb: W&B syncing is set to `offline` in this directory. Run `wandb online` or set WANDB_MODE=online to enable cloud syncing.
wandb: Run data is saved locally in /content/MKDT/wandb/offline-run-20260615_151138-8niqqitw
Test training data:  500
Number of images for pretrain: 50000
wandb: WARNING Data passed to `wandb.Image` should consist of values in the range [0, 255], image data will be normalized to this range, but behavior will be removed in a future version of wandb.
Pretraining LR:  0.1
encoding: 100% 2/2 [00:00<00:00,  3.70it/s]
torch.Size([500, 2048])
torch.Size([500])

L2 Regularization weight: 0.001
Loss: **** | Train Acc: ****% :   0% 0/20 [00:00<?, ?it/s]/content/MKDT/evaluation/linear_evaluation.py:33: UserWarning: Converting a tensor with requires_grad=True t